In [0]:
-- Create Quantexa Data Product organisations_all Schema metadata
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- metadata table
DROP TABLE IF EXISTS metadata;

CREATE OR REPLACE TABLE metadata ( 
	supplier STRING,
	data_product STRING,
	data_product_version STRING,
	schema_name STRING,
	schema_version STRING,
	schema_variant_name STRING,
	schema_variant_version STRING,
	instance STRING,
	instance_version STRING
 );

INSERT INTO metadata 
(
  supplier, 
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
) 
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.individual_customers',
  '0.1',
  'individual_customers.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);

In [0]:
-- Create Quantexa Data Product individual_customers Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- Organisation Customer table
DROP TABLE IF EXISTS hub_individual_customers;

CREATE OR REPLACE TABLE hub_individual_customers (
    individual_customer_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'Individual Customer Identity',
    individual_entity_id STRING,
    customer_entity_id STRING,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
		period_start TIMESTAMP,
		period_end TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE hub_individual_customers ADD CONSTRAINT pk_individualcustomers_hubindividualcustomers_individualcustomerid PRIMARY KEY (individual_customer_id);


In [0]:
-- Create Quantexa Data Product individual_customers Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- Individual Customer table
DROP TABLE IF EXISTS sat_individual_customers;

CREATE OR REPLACE TABLE sat_individual_customers (
    sat_individual_customer_id BIGINT NOT NULL COMMENT 'Individual Customer Identity',
    individual_customer_id BIGINT NOT NULL,
    address_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    record_source_id BIGINT NOT NULL DEFAULT 1,
		individual_id BIGINT NOT NULL,
	  customer_name STRING,
	  data_source_code STRING,
	  data_source_concept_id BIGINT,
	  data_source_raw_code STRING,
	  data_source_raw_concept_id BIGINT,
	  type_code STRING,
	  type_concept_id BIGINT,
	  type_raw_code STRING,
	  type_raw_concept_id BIGINT,
	  line_of_business_code STRING,
	  line_of_business_concept_id BIGINT,
	  line_of_business_raw_code STRING,
	  line_of_business_raw_concept_id BIGINT,
	  importance_code STRING,
	  importance_concept_id BIGINT,
	  importance_raw_code STRING,
	  importance_raw_concept_id BIGINT,
	  status_code STRING,
	  status_concept_id BIGINT,
	  status_raw_code STRING,
	  status_raw_concept_id BIGINT,
		risk_rating INT,
		risk_rating_raw STRING,
		risk_rating_code STRING,
		risk_rating_concept_id BIGINT,
		risk_rating_raw_code STRING,
		risk_rating_raw_concept_id BIGINT,
	  is_offboarded BOOLEAN,
	  is_customer_alarm BOOLEAN,
		period_start TIMESTAMP,
		period_end TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE sat_individual_customers ADD CONSTRAINT pk_individualcustomers_satindividualcustomers_satindividualcustomerid PRIMARY KEY (sat_individual_customer_id);
ALTER TABLE sat_individual_customers ADD CONSTRAINT fk_individualcustomers_satindividualcustomers_hubindividualcustomers_individualcustomerid FOREIGN KEY (individual_customer_id) REFERENCES hub_individual_customers(individual_customer_id);


In [0]:
-- Create Quantexa Data Product individual_customers Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- Individual Customer Risk Score table
DROP TABLE IF EXISTS sat_individual_customers_risk_score;

CREATE OR REPLACE TABLE sat_individual_customers_risk_score (
	individual_customer_risk_score_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'Individual Customer Risk Score Identity',
	individual_customer_id BIGINT NOT NULL,
  load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  record_source_id BIGINT NOT NULL DEFAULT 1,
	risk_code STRING,
	risk_concept_id BIGINT,
	risk_raw_code STRING,
	risk_raw_concept_id BIGINT,
	risk_score STRING,
	period_start TIMESTAMP,
	period_end TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE sat_individual_customers_risk_score ADD CONSTRAINT pk_individualcustomers_individualcustomersriskscore_individualcustomerriskscoreid PRIMARY KEY (individual_customer_risk_score_id);
ALTER TABLE sat_individual_customers_risk_score ADD CONSTRAINT fk_individualcustomers_individualcustomersriskscore_hubindividualcustomers_individualcustomerid FOREIGN KEY (individual_customer_id) REFERENCES hub_individual_customers(individual_customer_id);




In [0]:
-- Create Quantexa Data Product individual_customers Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- Individual Customer Warning Signals table
DROP TABLE IF EXISTS sat_individual_customers_warning_signals;

CREATE OR REPLACE TABLE sat_individual_customers_warning_signals (
	individual_customer_warning_signals_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'Individual Customer Warning Signals Identity',
	individual_customer_id BIGINT NOT NULL,
  load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  record_source_id BIGINT NOT NULL DEFAULT 1,
	warning_code STRING,
	warning_concept_id BIGINT,
	warning_raw_code STRING,
	warning_raw_concept_id BIGINT,
	warning_signal STRING,
	period_start TIMESTAMP,
	period_end TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE sat_individual_customers_warning_signals ADD CONSTRAINT pk_individualcustomesr_individualcustomerswarningsignals_individualcustomerwarningsignalsid PRIMARY KEY (individual_customer_warning_signals_id);
ALTER TABLE sat_individual_customers_warning_signals ADD CONSTRAINT fk_individualcustomers_individualcustomerswarningsignals_hubindividualcustomers_individualcustomerid FOREIGN KEY (individual_customer_id) REFERENCES hub_individual_customers(individual_customer_id);


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_individual_customers AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.individual_customer_id,
  s.customer_name,
  ihn.given1 AS individual_given_name,
  ihn.family AS individual_family_name,
  h.customer_entity_id, 
  h.individual_entity_id, 
  h.period_start as customer_period_start,
  h.period_end as customer_period_end,
  s.type_code as customer_type_code,
  s.status_code as customer_status_code,
  s.risk_rating,
  s.is_customer_alarm,
  s.is_offboarded,
  s.importance_code,
  s.line_of_business_code,
  s.risk_rating_code,
  hind.individual_id, 
	hind.national_id,
	hind.period_start AS hub_period_start,
	hind.period_end AS hub_period_end,
  sind.sat_individual_id,
	sind.birth_date,
	sind.birth_datetime,
	sind.birth_date_numeric,
	sind.estimated_birth_date,
	sind.estimated_birth_date_numeric,
	sind.is_multiple_birth,
	sind.multiple_birth_number,
	sind.birth_place_city,
	sind.birth_place_district,
	sind.birth_place_country,
	sind.birth_place_country_code,
	sind.photo,
	sind.is_deceased,
	sind.deceased_datetime,
	sind.deceased_date_numeric,
	sind.gender_code,
	sind.gender_concept_id,
	sind.gender_raw_code,
	sind.gender_raw_concept_id,
	sind.marital_status_code,
	sind.marital_status_concept_id,
	sind.marital_status_raw_code,
	sind.marital_status_raw_concept_id,
	sind.ethnicity_code,
	sind.ethnicity_concept_id,
	sind.ethnicity_raw_code,
	sind.ethnicity_raw_concept_id,
	sind.religion_code,
	sind.religion_concept_id,
	sind.religion_raw_code,
	sind.religion_raw_concept_id,
	sind.occupation_code,
	sind.occupation_concept_id,
	sind.occupation_raw_code,
	sind.occupation_raw_concept_id,
	sind.nationality_code,
	sind.nationality_concept_id,
	sind.nationality_raw_code,
	sind.nationality_raw_concept_id,
	sind.country_of_residence_code,
	sind.country_of_residence_concept_id,
	sind.country_of_residence_raw_code,
	sind.country_of_residence_raw_concept_id,
	sind.primary_address_id,
	sind.annual_income,
	sind.number_of_dependants,
	sind.graduation_date,
	sind.is_employee,
	sind.is_ex_employee,
	sind.is_graduate,
	sind.is_interpreter_required,
	sind.is_employed,
	sind.is_retired,
	sind.is_student,
	sind.is_individual_merged,
	sind.period_start,
	sind.period_end,
	sind.load_datetime,
	sind.record_source_id,
	sind.data_source_code,
	sind.data_source_concept_id,
	sind.data_source_raw_code,
	sind.data_source_raw_concept_id


    FROM demo_banking_silver.qdp_individual_customers.hub_individual_customers h
    JOIN demo_banking_silver.qdp_individual_customers.sat_individual_customers s
      ON h.individual_customer_id = s.individual_customer_id
    JOIN demo_banking_silver.qdp_individuals_all.hub_individual hind
      ON hind.individual_id = s.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_individual sind
      ON hind.individual_id = sind.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_human_name ihn 
      ON hind.individual_id = ihn.individual_id
    JOIN demo_banking_silver.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
    WHERE ihn.is_preferred = true    
;

SELECT * from view_individual_customers;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_individual_customer_addresses AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.individual_customer_id,
  s.customer_name,
  ihn.given1 AS individual_given_name,
  ihn.family AS individual_family_name,
  sadd.full_address as customer_full_address,
  sadd.postal_code as customer_full_address_postal_code,
  h.customer_entity_id, 
  h.individual_entity_id, 
  h.period_start as customer_period_start,
  h.period_end as customer_period_end,
  s.type_code as customer_type_code,
  s.status_code as customer_status_code,
  s.risk_rating,
  s.is_customer_alarm,
  s.is_offboarded,
  s.importance_code,
  s.line_of_business_code,
  s.risk_rating_code,
  hind.individual_id, 
	hind.national_id,
	hind.period_start AS hub_period_start,
	hind.period_end AS hub_period_end,
  sind.sat_individual_id,
	sind.birth_date,
	sind.birth_datetime,
	sind.estimated_birth_date,
	sind.is_multiple_birth,
	sind.multiple_birth_number,
	sind.birth_place_city,
	sind.birth_place_district,
	sind.birth_place_country,
	sind.birth_place_country_code,
	sind.is_deceased,
	sind.deceased_datetime,
	sind.gender_code,
	sind.marital_status_code,
	sind.ethnicity_code,
	sind.religion_code,
	sind.occupation_code,
	sind.nationality_code,
	sind.country_of_residence_code,
	sind.annual_income,
	sind.number_of_dependants,
	sind.graduation_date,
	sind.is_employee,
	sind.is_ex_employee,
	sind.is_graduate,
	sind.is_interpreter_required,
	sind.is_employed,
	sind.is_retired,
	sind.is_student,
	sind.period_start,
	sind.period_end


    FROM demo_banking_silver.qdp_individual_customers.hub_individual_customers h
    JOIN demo_banking_silver.qdp_individual_customers.sat_individual_customers s
      ON h.individual_customer_id = s.individual_customer_id
    JOIN demo_banking_silver.qdp_individuals_all.hub_individual hind
      ON hind.individual_id = s.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_individual sind
      ON hind.individual_id = sind.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_human_name ihn 
      ON hind.individual_id = ihn.individual_id
    JOIN demo_banking_silver.qdp_locations_all.hub_address hadd
      ON s.address_id = hadd.address_id
    JOIN demo_banking_silver.qdp_locations_all.sat_address sadd
      ON hadd.address_id = sadd.address_id
    JOIN demo_banking_silver.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
    WHERE ihn.is_preferred = true
    ORDER BY h.individual_customer_id ASC
;

SELECT * from view_individual_customer_addresses;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_individual_customer_contact_details AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.individual_customer_id,
  s.customer_name,
  ihn.given1 AS individual_given_name,
  ihn.family AS individual_family_name,
  icm.rank AS communication_method_rank,
  icm.is_primary_communication_method,
  icm.value AS contact_value,
  icm.type_code AS contact_type_code,
  icm.use_code AS contact_use_code,
  icm.telephony_country_code,
  icm.period_start AS contact_method_start,
  icm.period_end AS contact_method_end,
  sadd.full_address,
  sadd.postal_code,
  h.customer_entity_id, 
  h.individual_entity_id, 
  h.period_start as customer_period_start,
  h.period_end as customer_period_end,
  s.type_code as customer_type_code,
  s.status_code as customer_status_code,
  s.risk_rating,
  s.is_customer_alarm,
  s.is_offboarded,
  s.importance_code,
  s.line_of_business_code,
  s.risk_rating_code,
  hind.individual_id, 
	hind.national_id,
	hind.period_start AS hub_period_start,
	hind.period_end AS hub_period_end,
  sind.sat_individual_id,
	sind.birth_date,
	sind.birth_datetime,
	sind.estimated_birth_date,
	sind.is_multiple_birth,
	sind.multiple_birth_number,
	sind.birth_place_city,
	sind.birth_place_district,
	sind.birth_place_country,
	sind.birth_place_country_code,
	sind.is_deceased,
	sind.deceased_datetime,
	sind.gender_code,
	sind.marital_status_code,
	sind.ethnicity_code,
	sind.religion_code,
	sind.occupation_code,
	sind.nationality_code,
	sind.country_of_residence_code,
	sind.annual_income,
	sind.number_of_dependants,
	sind.graduation_date,
	sind.is_employee,
	sind.is_ex_employee,
	sind.is_graduate,
	sind.is_interpreter_required,
	sind.is_employed,
	sind.is_retired,
	sind.is_student,
	sind.period_start,
	sind.period_end


    FROM demo_banking_silver.qdp_individual_customers.hub_individual_customers h
    JOIN demo_banking_silver.qdp_individual_customers.sat_individual_customers s
      ON h.individual_customer_id = s.individual_customer_id
    JOIN demo_banking_silver.qdp_individuals_all.hub_individual hind
      ON hind.individual_id = s.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_individual sind
      ON hind.individual_id = sind.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_human_name ihn 
      ON hind.individual_id = ihn.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_communication_method icm 
      ON hind.individual_id = icm.individual_id
    JOIN demo_banking_silver.qdp_locations_all.hub_address hadd
      ON s.address_id = hadd.address_id
    JOIN demo_banking_silver.qdp_locations_all.sat_address sadd
      ON hadd.address_id = sadd.address_id
    JOIN demo_banking_silver.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
    WHERE ihn.is_preferred = true
    ORDER BY h.individual_customer_id ASC
;

SELECT * from view_individual_customer_contact_details;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_individual_customer_names AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.individual_customer_id,
  s.customer_name,
  h.customer_entity_id, 
  h.individual_entity_id, 
  h.period_start as customer_period_start,
  h.period_end as customer_period_end,
  s.type_code as customer_type_code,
  s.status_code as customer_status_code,
  s.risk_rating,
  s.is_customer_alarm,
  s.is_offboarded,
  s.importance_code,
  s.line_of_business_code,
  s.risk_rating_code,
  hind.individual_id, 
	hind.national_id,
	hind.period_start AS hub_period_start,
	hind.period_end AS hub_period_end,
  sind.sat_individual_id,
	sind.birth_date,
	sind.birth_datetime,
	sind.estimated_birth_date,
	sind.is_multiple_birth,
	sind.multiple_birth_number,
	sind.birth_place_city,
	sind.birth_place_district,
	sind.birth_place_country,
	sind.birth_place_country_code,
	sind.is_deceased,
	sind.deceased_datetime,
	sind.gender_code,
	sind.marital_status_code,
	sind.ethnicity_code,
	sind.religion_code,
	sind.occupation_code,
	sind.nationality_code,
	sind.country_of_residence_code,
	sind.annual_income,
	sind.number_of_dependants,
	sind.graduation_date,
	sind.is_employee,
	sind.is_ex_employee,
	sind.is_graduate,
	sind.is_interpreter_required,
	sind.is_employed,
	sind.is_retired,
	sind.is_student,
	sind.period_start,
	sind.period_end


    FROM demo_banking_silver.qdp_individual_customers.hub_individual_customers h
    JOIN demo_banking_silver.qdp_individual_customers.sat_individual_customers s
      ON h.individual_customer_id = s.individual_customer_id
    JOIN demo_banking_silver.qdp_individuals_all.hub_individual hind
      ON hind.individual_id = s.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_individual sind
      ON hind.individual_id = sind.individual_id
    JOIN demo_banking_silver.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
    ORDER BY h.individual_customer_id ASC  
;

SELECT * from view_individual_customer_names;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_individual_customer_accounts AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.individual_customer_id,
  s.customer_name,
  ihn.given1 AS individual_given_name,
  ihn.family AS individual_family_name,
	hacc.period_start AS bank_account_opened,
	hacc.period_end AS bank_account_closed,
  sacc.sort_code,
  sacc.account_number,
  sacc.account_name,
  sacc.iban,
  sacc.swiftbic,
  saddr.full_address AS bank_account_address,
  saddr.postal_code AS bank_account_postal_code,
  sacc.type_code,
  sacc.balance,
  sacc.overdraft_limit,
  sacc.loan_original_amount,
  sacc.loan_orininal_maturity_date,
  sacc.servicer_identification,
  sacc.servicer_scheme_name,    
  sacc.currency_code,
  sacc.branch_code,
  sacc.opened_datetime,
  sacc.closed_datetime,
  sacc.status_code,
  sacc.risk_rating as bank_account_risk_rating,
  sacc.risk_rating_code as bank_account_risk_rating_code,
  sacc.country_code,
  sacc.is_account_alarm,
  sacc.is_business_account,
  sacc.is_customer_acccount,
  sacc.is_counterparty_account,
  sacc.is_frozed,
  sacc.is_closed,
  sacc.is_open,
  sacc.is_excluded_from_monitoring,
  h.customer_entity_id, 
  h.individual_entity_id, 
  h.period_start as customer_period_start,
  h.period_end as customer_period_end,
  s.type_code as customer_type_code,
  s.status_code as customer_status_code,
  s.risk_rating as customer_risk_rating,
  s.is_customer_alarm,
  s.is_offboarded,
  s.importance_code,
  s.line_of_business_code,
  s.risk_rating_code as customer_risk_rting_code,
  hind.individual_id, 
	hind.national_id,
	hind.period_start AS hub_period_start,
	hind.period_end AS hub_period_end,
  sind.sat_individual_id,
	sind.birth_date,
	sind.birth_datetime,
	sind.estimated_birth_date,
	sind.is_multiple_birth,
	sind.multiple_birth_number,
	sind.birth_place_city,
	sind.birth_place_district,
	sind.birth_place_country,
	sind.birth_place_country_code,
	sind.is_deceased,
	sind.deceased_datetime,
	sind.gender_code,
	sind.marital_status_code,
	sind.ethnicity_code,
	sind.religion_code,
	sind.occupation_code,
	sind.nationality_code,
	sind.country_of_residence_code,
	sind.annual_income,
	sind.number_of_dependants,
	sind.graduation_date,
	sind.is_employee,
	sind.is_ex_employee,
	sind.is_graduate,
	sind.is_interpreter_required,
	sind.is_employed,
	sind.is_retired,
	sind.is_student,
	sind.period_start,
	sind.period_end


    FROM demo_banking_silver.qdp_individual_customers.hub_individual_customers h
    JOIN demo_banking_silver.qdp_individual_customers.sat_individual_customers s
      ON h.individual_customer_id = s.individual_customer_id
    JOIN demo_banking_silver.qdp_individuals_all.hub_individual hind
      ON hind.individual_id = s.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_individual sind
      ON hind.individual_id = sind.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_human_name ihn 
      ON hind.individual_id = ihn.individual_id
    JOIN demo_banking_silver.qdp_accounts_all.sat_account sacc 
      ON h.individual_customer_id = sacc.individual_customer_id
    JOIN demo_banking_silver.qdp_accounts_all.hub_account hacc 
      ON sacc.account_id = hacc.account_id
    JOIN demo_banking_silver.qdp_locations_all.sat_address saddr
      ON sacc.address_id = saddr.address_id
    JOIN demo_banking_silver.qdp_locations_all.hub_address haddr
      ON saddr.address_id = haddr.address_id
    JOIN demo_banking_silver.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
    WHERE ihn.is_preferred = true
    ORDER BY individual_customer_id
;

SELECT * from view_individual_customer_accounts;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_individual_customer_accounts_with_alarm AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.individual_customer_id,
  s.customer_name,
  ihn.given1 AS individual_given_name,
  ihn.family AS individual_family_name,
	hacc.period_start AS bank_account_opened,
	hacc.period_end AS bank_account_closed,
  sacc.sort_code,
  sacc.account_number,
  sacc.account_name,
  sacc.iban,
  sacc.swiftbic,
  saddr.full_address AS bank_account_address,
  saddr.postal_code AS bank_account_postal_code,
  sacc.type_code,
  sacc.balance,
  sacc.overdraft_limit,
  sacc.loan_original_amount,
  sacc.loan_orininal_maturity_date,
  sacc.servicer_identification,
  sacc.servicer_scheme_name,    
  sacc.currency_code,
  sacc.branch_code,
  sacc.opened_datetime,
  sacc.closed_datetime,
  sacc.status_code,
  sacc.risk_rating as bank_account_risk_rating,
  sacc.risk_rating_code as bank_account_risk_rating_code,
  sacc.country_code,
  sacc.is_account_alarm,
  sacc.is_business_account,
  sacc.is_customer_acccount,
  sacc.is_counterparty_account,
  sacc.is_frozed,
  sacc.is_closed,
  sacc.is_open,
  sacc.is_excluded_from_monitoring,
  h.customer_entity_id, 
  h.individual_entity_id, 
  h.period_start as customer_period_start,
  h.period_end as customer_period_end,
  s.type_code as customer_type_code,
  s.status_code as customer_status_code,
  s.risk_rating as customer_risk_rating,
  s.is_customer_alarm,
  s.is_offboarded,
  s.importance_code,
  s.line_of_business_code,
  s.risk_rating_code as customer_risk_rting_code,
  hind.individual_id, 
	hind.national_id,
	hind.period_start AS hub_period_start,
	hind.period_end AS hub_period_end,
  sind.sat_individual_id,
	sind.birth_date,
	sind.birth_datetime,
	sind.estimated_birth_date,
	sind.is_multiple_birth,
	sind.multiple_birth_number,
	sind.birth_place_city,
	sind.birth_place_district,
	sind.birth_place_country,
	sind.birth_place_country_code,
	sind.is_deceased,
	sind.deceased_datetime,
	sind.gender_code,
	sind.marital_status_code,
	sind.ethnicity_code,
	sind.religion_code,
	sind.occupation_code,
	sind.nationality_code,
	sind.country_of_residence_code,
	sind.annual_income,
	sind.number_of_dependants,
	sind.graduation_date,
	sind.is_employee,
	sind.is_ex_employee,
	sind.is_graduate,
	sind.is_interpreter_required,
	sind.is_employed,
	sind.is_retired,
	sind.is_student,
	sind.period_start,
	sind.period_end


    FROM demo_banking_silver.qdp_individual_customers.hub_individual_customers h
    JOIN demo_banking_silver.qdp_individual_customers.sat_individual_customers s
      ON h.individual_customer_id = s.individual_customer_id
    JOIN demo_banking_silver.qdp_individuals_all.hub_individual hind
      ON hind.individual_id = s.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_individual sind
      ON hind.individual_id = sind.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_human_name ihn 
      ON hind.individual_id = ihn.individual_id
    JOIN demo_banking_silver.qdp_accounts_all.sat_account sacc 
      ON h.individual_customer_id = sacc.individual_customer_id
    JOIN demo_banking_silver.qdp_accounts_all.hub_account hacc 
      ON sacc.account_id = hacc.account_id
    JOIN demo_banking_silver.qdp_locations_all.sat_address saddr
      ON sacc.address_id = saddr.address_id
    JOIN demo_banking_silver.qdp_locations_all.hub_address haddr
      ON saddr.address_id = haddr.address_id
    JOIN demo_banking_silver.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
    WHERE ihn.is_preferred = true AND sacc.is_account_alarm
    ORDER BY individual_customer_id
;

SELECT * from view_individual_customer_accounts_with_alarm;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_individual_customer_account_befeficiary_transactions AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.individual_customer_id,
  s.customer_name,
  ihn.given1 AS individual_given_name,
  ihn.family AS individual_family_name,
	hacc.period_start AS bank_account_opened,
	hacc.period_end AS bank_account_closed,
  sacc.sort_code,
  sacc.account_number,
  sacc.account_name,
  sacc.iban,
  sacc.swiftbic,
  saddr.full_address AS bank_account_address,
  saddr.postal_code AS bank_account_postal_code,
  sacc.type_code,
  sacc.balance,
  sacc.overdraft_limit,
  sacc.loan_original_amount,
  sacc.loan_orininal_maturity_date,
  sacc.servicer_identification,
  sacc.servicer_scheme_name,    
  sacc.currency_code,
  sacc.branch_code,
  sacc.opened_datetime,
  sacc.closed_datetime,
  sacc.status_code,
  sacc.risk_rating as bank_account_risk_rating,
  sacc.risk_rating_code as bank_account_risk_rating_code,
  sacc.country_code,
  sacc.is_account_alarm,
  sacc.is_business_account,
  sacc.is_customer_acccount,
  sacc.is_counterparty_account,
  sacc.is_frozed,
  sacc.is_closed,
  sacc.is_open,
  sacc.is_excluded_from_monitoring,
  h.customer_entity_id, 
  h.individual_entity_id, 
  h.period_start as customer_period_start,
  h.period_end as customer_period_end,
  s.type_code as customer_type_code,
  s.status_code as customer_status_code,
  s.risk_rating as customer_risk_rating,
  s.is_customer_alarm,
  s.is_offboarded,
  s.importance_code,
  s.line_of_business_code,
  s.risk_rating_code as customer_risk_rting_code,
  hind.individual_id, 
	hind.national_id,
	hind.period_start AS hub_period_start,
	hind.period_end AS hub_period_end,
  sind.sat_individual_id,
	sind.birth_date,
	sind.birth_datetime,
	sind.estimated_birth_date,
	sind.is_multiple_birth,
	sind.multiple_birth_number,
	sind.birth_place_city,
	sind.birth_place_district,
	sind.birth_place_country,
	sind.birth_place_country_code,
	sind.is_deceased,
	sind.deceased_datetime,
	sind.gender_code,
	sind.marital_status_code,
	sind.ethnicity_code,
	sind.religion_code,
	sind.occupation_code,
	sind.nationality_code,
	sind.country_of_residence_code,
	sind.annual_income,
	sind.number_of_dependants,
	sind.graduation_date,
	sind.is_employee,
	sind.is_ex_employee,
	sind.is_graduate,
	sind.is_interpreter_required,
	sind.is_employed,
	sind.is_retired,
	sind.is_student,
	sind.period_start,
	sind.period_end,
  htrans.transaction_id,
  htrans.transaction_entity_id,
  htrans.transaction_id_raw,
  htrans.period_start AS hub_trans_period_start,
  htrans.period_end AS hub_trans_period_end,
	strans.type_code AS trans_type_code,
	strans.type_concept_id AS trans_type_concept_id,
	strans.type_raw_code AS trans_type_raw_code,
	strans.type_raw_concept_id AS trans_type_raw_concept_id,
	strans.debit_or_credit_code,
	strans.debit_or_credit_concept_id,
	strans.debit_or_credit_raw_code,
	strans.debit_or_credit_raw_concept_id,
	strans.counterparty_account_id,
	strans.counterparty_account_transaction_id,
	strans.is_self_transaction,
	strans.is_international_transaction,
 	strans.amount,
	strans.datetime,
	strans.description,
	strans.balance_after,
	strans.balance_before,	
	strans.payment_method_code,
	strans.payment_method_concept_id,
	strans.payment_method_raw_code,
	strans.payment_method_raw_concept_id,
	strans.reference,
	strans.account_sort_code,
	strans.account_number AS trans_account_number,
	strans.account_number_suffix,
 	strans.iban AS trans_iban,
	strans.swiftbic AS trans_swiftbic,
	strans.beneficiary_name,
	strans.currency_code AS trans_currency_code,
	strans.currency_concept_id AS trans_currency_concept_id,
	strans.currency_raw_code AS trans_currency_raw_code,
	strans.currency_raw_concept_id AS trans_currency_raw_concept_id,
	strans.country_code AS trans_country_code,
	strans.country_concept_id AS trans_country_concept_id,
	strans.country_raw_code AS trans_country_raw_code,
	strans.country_raw_concept_id AS  trans_country_raw_concept_id,
	strans.original_amount,
	strans.original_account_number,
	strans.original_account_sort_code,
	strans.original_iban,
	strans.original_account_name,
	strans.original_currency_code,
	strans.original_currency_concept_id,
	strans.original_currency_raw_code,
	strans.original_currency_raw_concept_id,
	strans.original_country_code,
	strans.original_country_concept_id,
	strans.original_country_raw_code,
	strans.original_country_raw_concept_id,
	strans.original_bank,
	strans.original_bank_country_code,
	strans.institution_bank,
	strans.institution_bank_country_code,
	strans.sending_bank,
	strans.sending_bank_country_code,
	strans.beneficiary_bank,
	strans.beneficiary_bank_country_code,
	strans.payment_booking_date,
	strans.payment_booking_system_code,
	strans.payment_booking_system_concept_id,
	strans.payment_booking_system_raw_code,
	strans.payment_booking_system_raw_concept_id,
	strans.payment_type_code,
	strans.payment_type_concept_id,
	strans.payment_type_raw_code,
	strans.payment_type_raw_concept_id,
	strans.other_receiver_information,
	strans.payment_party_id,
	strans.payment_account_number,
	strans.payment_source_code,	
	strans.period_start AS sat_trans_period_start,
	strans.period_end AS sat_trans_period_end,
	strans.load_datetime AS trans_load_datetime,
	strans.record_source_id AS trans_record_source_id,
	strans.data_source_code AS trans_data_source_code,
	strans.data_source_concept_id AS trans_data_source_concept_id,
	strans.data_source_raw_code AS trans_data_source_raw_code,
	strans.data_source_raw_concept_id AS trans_data_source_raw_concept_id


    FROM demo_banking_silver.qdp_individual_customers.hub_individual_customers h
    JOIN demo_banking_silver.qdp_individual_customers.sat_individual_customers s
      ON h.individual_customer_id = s.individual_customer_id
    JOIN demo_banking_silver.qdp_individuals_all.hub_individual hind
      ON hind.individual_id = s.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_individual sind
      ON hind.individual_id = sind.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_human_name ihn 
      ON hind.individual_id = ihn.individual_id
    JOIN demo_banking_silver.qdp_accounts_all.sat_account sacc 
      ON h.individual_customer_id = sacc.individual_customer_id
    JOIN demo_banking_silver.qdp_accounts_all.hub_account hacc 
      ON sacc.account_id = hacc.account_id
    JOIN demo_banking_silver.qdp_locations_all.sat_address saddr
      ON sacc.address_id = saddr.address_id
    JOIN demo_banking_silver.qdp_locations_all.hub_address haddr
      ON saddr.address_id = haddr.address_id
    JOIN demo_banking_silver.qdp_transactions_all.hub_transaction htrans
      ON htrans.bene_account_id = hacc.account_id
    JOIN demo_banking_silver.qdp_transactions_all.sat_transaction strans
      ON htrans.transaction_id = strans.transaction_id
    JOIN demo_banking_silver.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
    WHERE ihn.is_preferred = true
    ORDER BY individual_customer_id
;

SELECT * from view_individual_customer_account_befeficiary_transactions;


